In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
from pandas.plotting import scatter_matrix
import seaborn as sns

from statsmodels.graphics.tsaplots import plot_acf

In [38]:
# Load Train dataset.
TRAIN_LOAD_LOCATION = '../data/raw/train.xlsx'

df = pd.read_excel(TRAIN_LOAD_LOCATION)

In [30]:
df.head()

,timestamp,Year,Month,Day,Hour,Load,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,Site-5 Temp,Site-1 GHI,Site-2 GHI,Site-3 GHI,Site-4 GHI,Site-5 GHI,avg_region_temp,avg_region_ghi,is_weekend
0,2020-01-01 00:00:00,2020,1,1,0,1997,46.40,46.76,41.54,48.92,46.58,0,0,0,0,0,46.040,0.0,0
1,2020-01-01 01:00:00,2020,1,1,1,1921,46.94,47.48,41.36,47.48,44.78,0,0,0,0,0,45.608,0.0,0
2,2020-01-01 02:00:00,2020,1,1,2,1861,46.58,47.84,41.18,47.66,43.16,0,0,0,0,0,45.284,0.0,0
3,2020-01-01 03:00:00,2020,1,1,3,1833,45.68,46.58,39.74,47.30,42.80,0,0,0,0,0,44.420,0.0,0
4,2020-01-01 04:00:00,2020,1,1,4,1847,45.14,45.50,39.20,47.48,44.42,0,0,0,0,0,44.348,0.0,0


In [40]:
def baseline_df_transformer(raw_data_df):

    YEARS_DICT = {1:2020, 2:2021, 3:2022}

    TEMP_COLS = [f"Site-{i + 1} Temp" for i in range(5)]
    GHI_COLS = [f"Site-{i + 1} GHI" for i in range(5)]

    df_transformed = raw_data_df.copy()

    # Transform temperature readings to Farenheit.
    df_transformed[TEMP_COLS] = df_transformed[TEMP_COLS].apply(lambda x: x * 9/5 + 32)
    
    # Average the temperature and GHI measurements.
    df_transformed["avg_region_temp"] = df_transformed[TEMP_COLS].mean(axis=1)
    df_transformed["avg_region_ghi"] = df_transformed[GHI_COLS].mean(axis=1)

    # Create timestamps.
    df_transformed['Year'] = df_transformed['Year'].map(YEARS_DICT)
    df_transformed['Hour'] = df_transformed['Hour'] - 1
    df_transformed['timestamp'] = pd.to_datetime(df_transformed[['Year', 'Month', 'Day', 'Hour']])

    df_transformed["Hour_sin"] = np.sin(2*np.pi*df_transformed["Hour"]/24)
    df_transformed["Hour_cos"] = np.cos(2*np.pi*df_transformed["Hour"]/24)
    
    df_transformed["Month_sin"] = np.sin(2*np.pi*df_transformed["Month"]/24)
    df_transformed["Month_cos"] = np.cos(2*np.pi*df_transformed["Month"]/24)

    # Ensure chronological ordering.
    df_transformed = df_transformed.sort_values("timestamp")

    # Add feature indicating weekend.
    df_transformed['is_weekend'] = (df_transformed['timestamp'].dt.weekday >= 5).astype(int)

    # Rearrange columns

    cols_to_keep = ["Load", "Month_sin", "Month_cos", "Day", "Hour_sin", "Hour_cos", "avg_region_temp", "avg_region_ghi"]
    df_transformed = df_transformed[["timestamp"] + cols_to_keep]

    return df_transformed
    

In [42]:
df = baseline_transformer(df)

In [44]:
df.head()

,timestamp,Year,Month,Day,Hour,Load,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,...,Site-3 GHI,Site-4 GHI,Site-5 GHI,avg_region_temp,avg_region_ghi,Hour_sin,Hour_cos,Month_sin,Month_cos,is_weekend
0,2020-01-01 00:00:00,2020,1,1,0,1997,46.40,46.76,41.54,48.92,...,0,0,0,46.040,0.0,0.000000,1.000000,0.258819,0.965926,0
1,2020-01-01 01:00:00,2020,1,1,1,1921,46.94,47.48,41.36,47.48,...,0,0,0,45.608,0.0,0.258819,0.965926,0.258819,0.965926,0
2,2020-01-01 02:00:00,2020,1,1,2,1861,46.58,47.84,41.18,47.66,...,0,0,0,45.284,0.0,0.500000,0.866025,0.258819,0.965926,0
3,2020-01-01 03:00:00,2020,1,1,3,1833,45.68,46.58,39.74,47.30,...,0,0,0,44.420,0.0,0.707107,0.707107,0.258819,0.965926,0
4,2020-01-01 04:00:00,2020,1,1,4,1847,45.14,45.50,39.20,47.48,...,0,0,0,44.348,0.0,0.866025,0.500000,0.258819,0.965926,0
